In [ ]:
#@title Mise en place de l'environnement
!pip install evidently googletrans-py
!git clone https://github.com/nzmonzmp/dataset-ames.git
import io
import matplotlib.pyplot as plt
import nltk
import numpy
import pandas
import random
import requests
import scipy.stats
import seaborn
import sklearn.ensemble
import sklearn.feature_extraction.text
import sklearn.linear_model
import sklearn.model_selection
import sklearn.pipeline
import warnings
import zipfile
warnings.filterwarnings('ignore')

nltk.download('words')
nltk.download('wordnet')
nltk.download('omw-1.4')

def preprocess(train_file, test_file):
  train_X = pandas.read_csv(train_file, index_col="Id")
  test_X = pandas.read_csv(test_file, index_col="Id")

  train_y = train_X.pop("SalePrice")

  all_X = pandas.concat([train_X, test_X])

  cols_1 = ["LotFrontage"]
  all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

  cols_2 = ["MSZoning", "Electrical", "KitchenQual", "Exterior1st",
            "Exterior2nd", "SaleType", "Utilities"]
  all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

  cols_4 = ["GarageYrBlt", "GarageArea", "GarageCars", "BsmtFinSF1",
            "BsmtFinSF2", "BsmtFullBath", "BsmtHalfBath", "BsmtUnfSF",
            "MasVnrArea", "TotalBsmtSF"]
  all_X[cols_4] = all_X[cols_4].fillna(0)

  cols_5 = ["Functional"]
  all_X[cols_5] = all_X[cols_5].fillna("Typ")

  all_X = all_X.fillna("NA")

  cols_numerical2label = ['MSSubClass']
  all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

  quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
  quality_columns = ["BsmtCond", "BsmtQual", "ExterCond", "ExterQual",
                      "FireplaceQu", "GarageCond", "GarageQual", "HeatingQC",
                      "KitchenQual", "PoolQC"]
  street_mapping = dict(NA=0, Grvl=1, Pave=2)
  bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

  replace_mapping = dict(
    Alley=street_mapping,
    BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
    BsmtFinType1=bsmt_fin_mapping,
    BsmtFinType2=bsmt_fin_mapping,
    Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
    LandSlope=dict(Sev=1, Mod=2, Gtl=3),
    LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
    PavedDrive=dict(NA=0, N=1, P=2, Y=3),
    Street=dict(Grvl=1, Pave=2),
    Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  )

  for quality_column in quality_columns:
    replace_mapping[quality_column] = quality_mapping

  all_X.replace(replace_mapping, inplace=True)

  print(f"Nombre de NAs : {all_X.isnull().sum().sum()}")

  return (all_X.iloc[:train_X.shape[0], :],
          train_y,
          all_X.iloc[train_X.shape[0]:, :])


def download_medecine_reviews() -> tuple[pandas.DataFrame, pandas.DataFrame]:
  """Data source: https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+%28Drugs.com%29

  Citation:
    Felix Gräßer, Surya Kallumadi, Hagen Malberg, and Sebastian Zaunseder.
    2018.
    Aspect-Based Sentiment Analysis of Drug Reviews Applying Cross-Domain and Cross-Data Learning.
    In Proceedings of the 2018 International Conference on Digital Health (DH '18).
    ACM, New York, NY, USA, 121-125.
  """
  content = requests.get(
      "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
  ).content
  with zipfile.ZipFile(io.BytesIO(content)) as arc:
      raw_data = pandas.read_csv(arc.open("drugsComTest_raw.tsv"), sep="\t")
  return raw_data[["drugName", "condition", "review",	"rating"]]


def filter_medecine_reviews(df: pandas.DataFrame, condition: str
                            ) -> pandas.DataFrame:
  df = df.loc[(df["condition"] == condition) & (df["rating"].isin([1, 10])),
                     ["review", "rating"]]
  df["is_positive"] = df["rating"].apply(
      lambda x: 0 if x == 1 else 1)
  return df.drop(columns="rating")


def split_medecine_reviews(df: pandas.DataFrame
                           ) -> tuple[pandas.DataFrame, pandas.DataFrame]:
  X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
      df['review'],
      df['is_positive'],
      test_size=0.4,
      random_state=42,
      shuffle=True)

  reference = pandas.DataFrame({'review': X_train, 'is_positive': y_train})
  valid = pandas.DataFrame({'review': X_test, 'is_positive': y_test})
  return reference, valid

# Monitoring — Dérive des données et des concepts

## Prise en main de la bibliothèque `evidently`

### Chargement des données

In [ ]:
reference_df, _, current_df = preprocess("dataset-ames/train.csv", "dataset-ames/test.csv")

### Imports

In [ ]:
from evidently import (
    BinaryClassification,
    Dataset,
    DataDefinition,
    Report,
)

from evidently.descriptors import (
    NonLetterCharacterPercentage,
    OOVWordsPercentage,
    SentenceCount,
    TextLength,
    WordCount,
)

from evidently.metrics import MissingValueCount, UniqueValueCount

from evidently.presets import (
    ClassificationPreset,
    DataDriftPreset,
    ValueStats,
)

### Création d'un `Dataset` muni d'une `DataDefinition`

En suivant les étapes décrites dans la [documentation evidently](https://docs.evidentlyai.com/docs/library/data_definition), créez deux datasets, l'un,  `reference`, pour `reference_df` et l'autre, `current`, pour `current_df`. Vous pourrez utiliser la `DataDefinition` par défaut.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
reference = Dataset.from_pandas(reference_df, data_definition=DataDefinition())
current = Dataset.from_pandas(current_df, data_definition=DataDefinition())

### Rapports

Evidently peut générer des rapports & des suites de tests.

Les rapports sont destinés à être lus et étudiés par des humains, là où les suites de tests sont plus destinés à l'automatisation, par exemple pour déclencher un réapprentissage automatique.

Commencez par créer un tout premier rapport qui utilise les [paramètres par défaut pour détecter la dérive des données](https://docs.evidentlyai.com/metrics/preset_data_drift).

In [ ]:
# Votre code ici

#### Solution

In [ ]:
report = Report([DataDriftPreset()])
run = report.run(current_data=current, reference_data=reference)
run

### Étudier des colonnes en particulier

Dans le jeu de données que nous utilisons pour ces travaux pratiques (AMES), deux colonnes sont particulièrement importantes&nbsp; `OverallQual` & `GrLivArea`.

Créez un rapport sur ces deux colonnes avec la métrique `UniqueValueCount` pour la colonne `OverallQual` & le preset `ValueStats` pour la colonne `GrLivArea`.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
report = Report([UniqueValueCount(column="OverallQual"), ValueStats("GrLivArea")])
run = report.run(current_data=current, reference_data=reference)
run

### Sauvegarde d'un rapport

Vous pouvez [sauvegarder un rapport](https://docs.evidentlyai.com/docs/library/output_formats), au format HTML pour être lu directement ou JSON pour être exploité par d'autres programmes.

Sauvegardez le rapport de dérive des données au format JSON.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
report = Report([DataDriftPreset()])
eval = report.run(reference_data=reference, current_data=current)
eval.save_json('data-drift-report.json')

### Suites de tests

Les [suites de tests](https://docs.evidentlyai.com/docs/library/tests) sont plus adaptées que les rapports à un contexte automatisé comme la CI/CD.

Pour commencer, créez une première suite de tests qui utilisera le preset `DataDriftPreset` que nous avons utilisé plus haut pour produire un rapport.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
report = Report([DataDriftPreset()], include_tests=True)
run = report.run(reference_data=reference, current_data=current)
run

### Analyse des résultats en Python

Les tests étant plus utilisés pour l'automatisation, on analyse régulièrement leur résultat en Python. Calculez le pourcentage de tests réussis à partir de la dernière suite de tests. Vous pourrez vous aider de la méthode `dict`.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
results = run.dict()
successes = sum(d["status"] == "SUCCESS" for d in results["tests"])
percentage = successes / len(results["tests"]) * 100
percentage

## Application de la bibliothèque `evidently` à des données textuelles

### Chargement des données

Les données que nous allons utiliser sont des avis à propos de médicaments traitant plusieurs pathologies. Nous allons pour l'instant nous intéresser aux médicaments traitant la douleur, *Pain* dans les données.

In [ ]:
raw_data = download_medecine_reviews()
filtered_data = filter_medecine_reviews(raw_data, "Pain")
reference_df, current_df = split_medecine_reviews(filtered_data)

In [ ]:
reference_df

### Entraînement d'un modèle de classification

In [ ]:
pipeline = sklearn.pipeline.Pipeline(
    [
        ("vectorization",
         sklearn.feature_extraction.text.TfidfVectorizer(
             sublinear_tf=True,
             max_df=0.5,
             stop_words="english")),
        ("classification",
         sklearn.linear_model.SGDClassifier(
             alpha=0.0001,
             max_iter=50,
             penalty='l1',
             loss='modified_huber',
             random_state=42))
    ])
pipeline.fit(reference_df['review'].values, reference_df['is_positive'].values)

Création d'une nouvelle colonne dans `reference_df` & `current_df` qui contient les prédictions :

In [ ]:
reference_df['predictions'] = pipeline.predict(reference_df['review'].values)
current_df['predictions'] = pipeline.predict(current_df['review'].values)

In [ ]:
reference_df

Définissez deux `Dataset` (`reference` & `current`) en utilisant une `DataDefinition` adaptée aux trois colonnes `review`, `is_positive` & `predictions` de nos données.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
data_definition = DataDefinition(
    text_columns=["review"],
    classification=[BinaryClassification(target="is_positive",
                                         prediction_labels="predictions")]
)

reference = Dataset.from_pandas(reference_df, data_definition=data_definition)
current = Dataset.from_pandas(current_df, data_definition=data_definition)

### Rapport de qualité de classification

Créez un rapport de [qualité de classification](https://docs.evidentlyai.com/metrics/preset_classification) à partir des données de référence et de validation.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
report = Report([ClassificationPreset()])

run = report.run(reference_data=reference, current_data=current)
run

### Détection d'une dérive « technique » des données

Nous allons simuler un événement courant dans une chaîne de traitement : un bug entraîne des traitements de mauvaise qualité. Ici nous allons même en simuler deux :

- Bug dans le nettoyage des balises HTML pendant le prétraitement des avis
- Bug de récupération des données entraînant des données dans une langue différente de la langue d'entraînement

In [ ]:
from googletrans import Translator
translator = Translator()

def translate_str(s):
  return translator.translate(s, dest='fr').text

random_html_tags = ('<body>, </body>', '<html><body>', '</body></html>', '<h1>', '</h1>',
                    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 0 0" width="0" height="0" focusable="false" role="none" style="visibility: hidden; position: absolute; left: -9999px; overflow: hidden;"><defs><filter id="wp-duotone-magenta-yellow"><feColorMatrix color-interpolation-filters="sRGB" type="matrix" values=" .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 "></feColorMatrix><feComponentTransfer color-interpolation-filters="sRGB"><feFuncR type="table" tableValues="0.78039215686275 1"></feFuncR><feFuncG type="table" tableValues="0 0.94901960784314"></feFuncG><feFuncB type="table" tableValues="0.35294117647059 0.47058823529412"></feFuncB><feFuncA type="table" tableValues="1 1"></feFuncA></feComponentTransfer><feComposite in2="SourceGraphic" operator="in"></feComposite></filter></defs></svg>')

def inject_random_html_tags(s):
  num_tags = 25
  for i in range(num_tags):
    random.seed(i)
    pos = random.choice(range(len(s)))
    s = s[:pos] + random.choice(random_html_tags) + s[pos:]

  return s

In [ ]:
current_disturbed_df = current_df[['review', 'is_positive']].copy()

In [ ]:
disturbed_num = int(len(current_disturbed_df) * 0.5)
random.seed(42)
disturbed_ind = random.sample(list(current_disturbed_df.index), k=disturbed_num)
current_disturbed_df.loc[disturbed_ind[:int(disturbed_num / 10)], 'review'] = \
current_disturbed_df.loc[disturbed_ind[:int(disturbed_num / 10)], 'review'].apply(inject_random_html_tags)
# current_disturbed_df.loc[disturbed_ind[int(disturbed_num / 10):], 'review'] = \
# current_disturbed_df.loc[disturbed_ind[int(disturbed_num / 10):], 'review'].apply(translate_str)

In [ ]:
current_disturbed_df['predictions'] = pipeline.predict(current_disturbed_df['review'].values)
current_disturbed = Dataset.from_pandas(current_disturbed_df, data_definition=data_definition)

### Production d'un nouveau rapport de qualité

Reprenez le code précédent pour analyser les performances du modèle sur ces données détériorées, en comparant cette fois aux données de validation « propres ».

In [ ]:
# Votre code ici

#### Solution

In [ ]:
report = Report([ClassificationPreset()])

run = report.run(reference_data=current, current_data=current_disturbed)
run

### Analyse des mauvaises performances du modèle

Produisez un rapport de dérive des données qui présentera la dérive des colonnes `is_positive` et `predictions` ainsi que la dérive des [descripteurs](https://docs.evidentlyai.com/docs/library/descriptors) textuels pour la colonne `review`. Il faudra au préalable définir une liste de descripteurs textuels à utiliser et les ajouter aux datasets déjà définis avec la méthode `add_descriptors`.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
descriptors = [
    NonLetterCharacterPercentage("review", alias="non_letters"),
    OOVWordsPercentage("review", alias="oov"),
    SentenceCount("review", alias="sentence_count"),
    TextLength("review", alias="text_length"),
    WordCount("review", alias="word_count"),
]

reference.add_descriptors(descriptors)
current.add_descriptors(descriptors)
current_disturbed.add_descriptors(descriptors)

report = Report([DataDriftPreset()])

run = report.run(reference_data=current, current_data=current_disturbed)
run

### Inspection manuelle des exemples fautifs

On peut constater l'augmentation des textes longs et de la présence de mots hors vocabulaire (*OOV*).

Utilisez l'export des descripteurs en dataframe pour observer les exemples fautifs, par exemple :

- Les avis qui ont une longueur supérieure à 1000 caractères
- Les avis qui ont plus de 30% des mots hors du vocabulaire. Comment `evidently` définit le vocabulaire ?

In [ ]:
# Votre code ici

#### Solution

In [ ]:
descriptors_df = current_disturbed.as_dataframe()
descriptors_df

In [ ]:
descriptors_df[descriptors_df['text_length'] > 1000]

In [ ]:
descriptors_df[descriptors_df['oov'] > 30]

### Dérive des données

Simulons maintenant une dérive des données en regardant une autre partie de nos données originales : les avis sur les médicaments pour traiter la dépression.

In [ ]:
new_content_df = filter_medecine_reviews(raw_data, "Depression")
new_content_df

In [ ]:
new_content_df["predictions"] = pipeline.predict(new_content_df.review.values)

### Rapport de qualité de classification

Comme précédemment, utilisez `evidently` pour quantifier l'évolution des performances du modèle.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
new_content = Dataset.from_pandas(new_content_df,
                                  data_definition=data_definition,
                                  descriptors=descriptors)

report = Report([ClassificationPreset()])

run = report.run(reference_data=current, current_data=new_content)
run

### Détection d'une dérive des données

Sans surprise, les performances sont très dégradées. Produisez un rapport de dérive des données. `evidently` aurait-il détecté cette dérive à temps pour permettre un réapprentissage ?

In [ ]:
# Votre code ici

#### Solution

In [ ]:
report = Report([DataDriftPreset()])

run = report.run(reference_data=current, current_data=new_content)
run